# 04 - Explainable AI SHAP & Integrated Gradients

Gi?i th?ch ??ng model Colab ?? train ? Notebook 02 b?ng SHAP v? Integrated Gradients.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
os.makedirs(PROJECT_PATH, exist_ok=True)
%cd {PROJECT_PATH}
print('PROJECT_PATH =', PROJECT_PATH)


In [ ]:
!pip -q install pandas numpy tensorflow shap pyarrow matplotlib


In [ ]:

import os, json, numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt, shap
from tensorflow.keras.models import load_model
OUT=f'{PROJECT_PATH}/data/colab_processed'; RESULTS=f'{PROJECT_PATH}/results/xai'; MODELS=f'{PROJECT_PATH}/models'; os.makedirs(RESULTS,exist_ok=True)
model_path=f'{MODELS}/colab_cyberdetect_mlp.h5'; assert os.path.exists(model_path), 'Run Notebook 02 first.'
meta=json.load(open(f'{MODELS}/colab_preprocessing_metadata.json',encoding='utf-8'))
features=meta['top_features']; X_train=pd.read_parquet(f'{OUT}/X_train_top30.parquet'); X_test=pd.read_parquet(f'{OUT}/X_test_top30.parquet')
model=load_model(model_path); background=X_train.values[:100]; samples=X_test.values[:50]
explainer=shap.GradientExplainer(model, background); sv=explainer.shap_values(samples); arr=sv[0] if isinstance(sv,list) else sv
if arr.ndim==3: arr=arr[:,:,0]
shap.summary_plot(arr, pd.DataFrame(samples,columns=features), show=False); plt.savefig(f'{RESULTS}/colab_shap_summary.png',bbox_inches='tight',dpi=150); plt.show()
x=tf.cast(samples[:1],tf.float32); baseline=tf.zeros_like(x); alphas=tf.reshape(tf.linspace(0.0,1.0,51),(-1,1)); interp=baseline+alphas*(x-baseline)
with tf.GradientTape() as tape:
    tape.watch(interp); pred=model(interp); target=tf.argmax(model(x)[0]); score=pred[:,target]
grads=tape.gradient(score,interp); ig=(x-baseline)*tf.reduce_mean(grads,axis=0); imp=np.abs(ig.numpy().ravel()); top=np.argsort(imp)[::-1][:20]
plt.figure(figsize=(9,7)); plt.barh([features[i] for i in top][::-1], imp[top][::-1]); plt.title('Integrated Gradients - Top Features'); plt.tight_layout(); plt.savefig(f'{RESULTS}/colab_integrated_gradients.png',dpi=150); plt.show()
